# A13 Modality Ablation — Statistical Analysis

Self-contained verification notebook. All per-subject accuracies are hard-coded from the
nine `A13_*_final_results.json` files (harness timestamp `20260728_*`), so this runs on a
laptop with no cluster access and no file transfer.

**Design.** 3 conditions (`both` / `emg_only` / `imu_only`) x 3 vocabulary sizes
(n_way = 3, 5, 10), 1-shot, fixed 24/4/4 split, seed 42, 500 test episodes per cell.
Four held-out test subjects, measured under every condition -> fully within-subject,
n = 4 experimental units.

**Analyses.**
1. Sanity checks against the reported JSON values
2. Descriptives
3. Two-way repeated-measures ANOVA (omnibus)
4. Greenhouse-Geisser sensitivity
5. Planned contrasts vs. the fused control, with CIs and effect sizes
6. Multiplicity correction under two family definitions
7. Why Wilcoxon signed-rank is unusable at n = 4
8. Minimum detectable effect
9. Validation-vs-test comparison

Requires `numpy pandas scipy statsmodels`. `pingouin` is optional (used only for the
Greenhouse-Geisser correction in section 4).

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM

pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

try:
    import pingouin as pg
    HAVE_PINGOUIN = True
except ImportError:
    HAVE_PINGOUIN = False
    print("pingouin not installed - section 4 will be skipped.")
    print("Install with:  pip install pingouin")

print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", stats.__name__)

numpy 1.24.3 | pandas 2.3.3 | scipy scipy.stats


## 1. Data and sanity checks

Per-subject accuracies transcribed from `test_results.per_user_acc` in each results file.
`REPORTED` holds the top-level `test_acc` and `best_val_acc` so we can confirm the
transcription reproduces them.

In [2]:
SUBJECTS = ["P004", "P104", "P105", "P121"]
CONDITIONS = ["both", "emg_only", "imu_only"]
N_WAYS = [3, 5, 10]

# test_results.per_user_acc, keyed (n_way, condition) -> [P004, P104, P105, P121]
PER_SUBJECT = {
    (3,  "both"):     [0.9247407407407406, 0.8401481481481482, 0.8592592592592593, 0.8790370370370371],
    (3,  "emg_only"): [0.9628888888888889, 0.8742962962962963, 0.8702962962962963, 0.8419259259259260],
    (3,  "imu_only"): [0.9258518518518519, 0.8407407407407409, 0.9008888888888890, 0.9494814814814815],
    (5,  "both"):     [0.8757777777777779, 0.7660000000000000, 0.7729777777777778, 0.7970222222222223],
    (5,  "emg_only"): [0.8562666666666667, 0.6573333333333332, 0.6909777777777778, 0.6669333333333333],
    (5,  "imu_only"): [0.9193777777777778, 0.7120888888888889, 0.7967111111111113, 0.8310666666666666],
    (10, "both"):     [0.7208444444444444, 0.6123999999999999, 0.5539333333333334, 0.6452222222222221],
    (10, "emg_only"): [0.5652888888888888, 0.4454000000000000, 0.4271555555555556, 0.3690444444444444],
    (10, "imu_only"): [0.7264888888888889, 0.5249111111111110, 0.6278222222222222, 0.7352666666666666],
}

# (test_acc, best_val_acc) as written in the JSONs
REPORTED = {
    (3,  "both"):     (0.8757962962962962, 0.8737962962962963),
    (3,  "emg_only"): (0.8873518518518520, 0.7875000000000000),
    (3,  "imu_only"): (0.9042407407407408, 0.8576851851851852),
    (5,  "both"):     (0.8029444444444445, 0.7891666666666667),
    (5,  "emg_only"): (0.7178777777777777, 0.6602500000000000),
    (5,  "imu_only"): (0.8148111111111112, 0.7544166666666666),
    (10, "both"):     (0.6331000000000000, 0.6165277777777778),
    (10, "emg_only"): (0.4517222222222221, 0.4189027777777778),
    (10, "imu_only"): (0.6536222222222221, 0.5968055555555556),
}

# mask_check energies -> confirms conditions saw the same episodes in the same order
MASK_ENERGY = {
    (3,  "both"):     (1759.98388671875,    5106.91015625),
    (3,  "emg_only"): (1759.98388671875,       0.0),
    (3,  "imu_only"): (   0.0,              5106.91015625),
    (5,  "both"):     (2699.572509765625,   9622.9013671875),
    (5,  "emg_only"): (2699.572509765625,      0.0),
    (5,  "imu_only"): (   0.0,              9622.9013671875),
    (10, "both"):     (5576.28857421875,   20704.9609375),
    (10, "emg_only"): (5576.28857421875,       0.0),
    (10, "imu_only"): (   0.0,             20704.9609375),
}

CHANCE = {3: 1/3, 5: 1/5, 10: 1/10}

In [3]:
# --- Check 1: the unweighted mean of per-subject accuracies reproduces test_acc ---
print("Check 1: mean(per_user_acc) == reported test_acc")
ok = True
for key, vals in PER_SUBJECT.items():
    recomputed, reported = np.mean(vals), REPORTED[key][0]
    match = np.isclose(recomputed, reported, atol=1e-12)
    ok &= match
    print(f"  {key[0]:>2}-way {key[1]:9s}  {recomputed:.16f} vs {reported:.16f}  {'OK' if match else 'MISMATCH'}")
assert ok, "transcription error"
print("\n  -> reported test_acc is the MACRO average (unweighted mean over subjects).\n")

# --- Check 2: masked-out modality energy is exactly zero ---
print("Check 2: masking is clean")
for (n, c), (e_emg, e_imu) in MASK_ENERGY.items():
    if c == "emg_only": assert e_imu == 0.0
    if c == "imu_only": assert e_emg == 0.0
print("  -> every masked block has exactly zero energy.\n")

# --- Check 3: unimodal energies match the fused control at the same n_way ---
print("Check 3: conditions saw identical episodes at matched n_way")
for n in N_WAYS:
    b_emg, b_imu = MASK_ENERGY[(n, "both")]
    assert MASK_ENERGY[(n, "emg_only")][0] == b_emg
    assert MASK_ENERGY[(n, "imu_only")][1] == b_imu
    print(f"  {n:>2}-way  EMG energy {b_emg:>10.2f} shared | IMU energy {b_imu:>10.2f} shared")
print("\n  -> the comparison is properly matched; differences are not a sampling artifact.")

Check 1: mean(per_user_acc) == reported test_acc
   3-way both       0.8757962962962962 vs 0.8757962962962962  OK
   3-way emg_only   0.8873518518518519 vs 0.8873518518518519  OK
   3-way imu_only   0.9042407407407408 vs 0.9042407407407408  OK
   5-way both       0.8029444444444445 vs 0.8029444444444445  OK
   5-way emg_only   0.7178777777777777 vs 0.7178777777777777  OK
   5-way imu_only   0.8148111111111112 vs 0.8148111111111112  OK
  10-way both       0.6331000000000000 vs 0.6331000000000000  OK
  10-way emg_only   0.4517222222222222 vs 0.4517222222222221  OK
  10-way imu_only   0.6536222222222221 vs 0.6536222222222221  OK

  -> reported test_acc is the MACRO average (unweighted mean over subjects).

Check 2: masking is clean
  -> every masked block has exactly zero energy.

Check 3: conditions saw identical episodes at matched n_way
   3-way  EMG energy    1759.98 shared | IMU energy    5106.91 shared
   5-way  EMG energy    2699.57 shared | IMU energy    9622.90 shared
  10-way  E

In [4]:
# Long-format frame: 3 conditions x 3 n_way x 4 subjects = 36 rows
df = pd.DataFrame([
    dict(subject=s, n_way=n, cond=c, acc=v * 100)
    for (n, c), vals in PER_SUBJECT.items()
    for s, v in zip(SUBJECTS, vals)
])
print(df.shape)
df.head(8)

(36, 4)


,subject,n_way,cond,acc
0,P004,3,both,92.4741
1,P104,3,both,84.0148
2,P105,3,both,85.9259
3,P121,3,both,87.9037
4,P004,3,emg_only,96.2889
5,P104,3,emg_only,87.4296
6,P105,3,emg_only,87.0296
7,P121,3,emg_only,84.1926


## 2. Descriptives

`sd` is the **sample** standard deviation across the four subjects (ddof = 1). Note that
`std_acc` in the JSON files uses the population standard deviation (ddof = 0) and is
therefore smaller; the sample value is the correct one for inference.

In [5]:
desc = (df.groupby(["n_way", "cond"])["acc"]
          .agg(mean="mean", sd=lambda x: x.std(ddof=1), sd_pop=lambda x: x.std(ddof=0))
          .reset_index())
desc["chance"] = desc["n_way"].map(lambda n: CHANCE[n] * 100)

print("Test accuracy (%), mean over 4 subjects\n")
print(desc.pivot(index="n_way", columns="cond", values="mean").round(2).to_string())
print("\nSample SD across subjects (%)\n")
print(desc.pivot(index="n_way", columns="cond", values="sd").round(2).to_string())
print("\nPer-subject detail\n")
print(df.pivot_table(index=["n_way", "cond"], columns="subject", values="acc").round(2).to_string())

Test accuracy (%), mean over 4 subjects

cond     both  emg_only  imu_only
n_way                            
3     87.5800   88.7400   90.4200
5     80.2900   71.7900   81.4800
10    63.3100   45.1700   65.3600

Sample SD across subjects (%)

cond    both  emg_only  imu_only
n_way                           
3     3.6300    5.2400    4.6800
5     5.0300    9.3300    8.5800
10    6.9600    8.2400    9.8700

Per-subject detail

subject           P004    P104    P105    P121
n_way cond                                    
3     both     92.4700 84.0100 85.9300 87.9000
      emg_only 96.2900 87.4300 87.0300 84.1900
      imu_only 92.5900 84.0700 90.0900 94.9500
5     both     87.5800 76.6000 77.3000 79.7000
      emg_only 85.6300 65.7300 69.1000 66.6900
      imu_only 91.9400 71.2100 79.6700 83.1100
10    both     72.0800 61.2400 55.3900 64.5200
      emg_only 56.5300 44.5400 42.7200 36.9000
      imu_only 72.6500 52.4900 62.7800 73.5300


## 3. Two-way repeated-measures ANOVA (omnibus)

Every subject is measured under all 9 cells, so condition and n_way are both within-subject
factors and subject is the repeated unit. This is the omnibus test; it comes **before** any
pairwise comparison.

The effect to watch is the **interaction**. If condition x n_way is significant, the modality
effect depends on vocabulary size, and collapsing across n_way would be misleading — which is
what justifies the per-n_way decomposition in section 5.

In [6]:
aov = AnovaRM(df, depvar="acc", subject="subject", within=["cond", "n_way"]).fit()
print(aov)

tbl = aov.anova_table
print("\nInterpretation:")
for effect in tbl.index:
    p = tbl.loc[effect, "Pr > F"]
    verdict = "significant" if p < 0.05 else "not significant"
    print(f"  {effect:12s} F({tbl.loc[effect,'Num DF']:.0f},{tbl.loc[effect,'Den DF']:.0f})"
          f" = {tbl.loc[effect,'F Value']:7.2f}, p = {p:.4f}  -> {verdict}")

                  Anova
           F Value  Num DF  Den DF Pr > F
-----------------------------------------
cond         7.0966 2.0000  6.0000 0.0262
n_way      209.4114 2.0000  6.0000 0.0000
cond:n_way  19.2331 4.0000 12.0000 0.0000


Interpretation:
  cond         F(2,6) =    7.10, p = 0.0262  -> significant
  n_way        F(2,6) =  209.41, p = 0.0000  -> significant
  cond:n_way   F(4,12) =   19.23, p = 0.0000  -> significant


## 4. Greenhouse-Geisser sensitivity

RM-ANOVA with 3-level factors assumes sphericity. At n = 4 sphericity cannot be meaningfully
assessed, so the uncorrected p-values above are optimistic. Greenhouse-Geisser shrinks the
degrees of freedom by epsilon and inflates p. Any effect that survives GG is safe; any effect
that does not should be reported with the caveat.

In [7]:
if HAVE_PINGOUIN:
    aov_pg = pg.rm_anova(data=df, dv="acc", within=["cond", "n_way"],
                         subject="subject", correction=True, detailed=True)

    # pingouin has used hyphenated and underscored column names across versions
    ren = {"p-unc": "p_unc", "p-GG-corr": "p_GG_corr"}
    aov_pg = aov_pg.rename(columns={k: v for k, v in ren.items() if k in aov_pg.columns})
    cols = [c for c in ["Source", "ddof1", "ddof2", "F", "p_unc", "p_GG_corr", "eps", "ng2"]
            if c in aov_pg.columns]
    print(aov_pg[cols].to_string(index=False))

    print("")
    print("  p_unc     = uncorrected")
    print("  p_GG_corr = Greenhouse-Geisser corrected")
    print("  eps       = GG epsilon (1.0 = sphericity holds perfectly)")
    print("  ng2       = generalised eta-squared")
    print("")

    for r in aov_pg.itertuples():
        keeps = r.p_unc < 0.05 and r.p_GG_corr < 0.05
        loses = r.p_unc < 0.05 and r.p_GG_corr >= 0.05
        verdict = ("survives GG" if keeps else
                   "LOSES significance under GG" if loses else "n.s. either way")
        print(f"  {r.Source:12s} p_unc = {r.p_unc:.4f} -> p_GG = {r.p_GG_corr:.4f}   {verdict}")

    print("")
    print("  NOTE: pingouin warns that epsilon is approximate in two-way designs where")
    print("  both factors have more than 2 levels, which is the case here. Treat the")
    print("  corrected values as a sensitivity check, not a definitive test.")
else:
    print("pingouin not available - skipping. Uncorrected results in section 3 stand,")
    print("but should be described as uncorrected for sphericity.")

      Source  ddof1  ddof2        F  p_unc  p_GG_corr    eps    ng2
        cond      2      6   7.0966 0.0262     0.0570 0.6319 0.3504
       n_way      2      6 209.4114 0.0000     0.0001 0.6732 0.8099
cond * n_way      4     12  19.2331 0.0000     0.0238 0.2405 0.2569

  p_unc     = uncorrected
  p_GG_corr = Greenhouse-Geisser corrected
  eps       = GG epsilon (1.0 = sphericity holds perfectly)
  ng2       = generalised eta-squared

  cond         p_unc = 0.0262 -> p_GG = 0.0570   LOSES significance under GG
  n_way        p_unc = 0.0000 -> p_GG = 0.0001   survives GG
  cond * n_way p_unc = 0.0000 -> p_GG = 0.0238   survives GG

  NOTE: pingouin warns that epsilon is approximate in two-way designs where
  both factors have more than 2 levels, which is the case here. Treat the
  corrected values as a sensitivity check, not a definitive test.


c:\Users\kdmen\anaconda3\envs\fl_torch\lib\site-packages\pingouin\distribution.py:514: UserWarning: Epsilon values might be innaccurate in two-way repeated measures design where each  factor has more than 2 levels. Please  double-check your results.
  warnings.warn(


## 5. Planned contrasts against the fused control

Two contrasts per vocabulary size, both against the same control (`both`). This is a
many-to-one / Dunnett structure, not all-pairs — there is no `emg_only` vs `imu_only`
comparison because nobody asked one.

These contrasts were specified by the reviewers' question before any number was seen, so they
are **planned**, not post hoc in the problematic sense. The significant interaction in
section 3 independently licenses decomposing by n_way.

Reported per contrast: mean paired difference, 95% CI, Cohen's *d_z*, and the raw p-value.

In [8]:
def paired_contrast(n_way, cond, control="both"):
    """Paired comparison of `cond` against `control` across subjects, in percentage points."""
    a = np.array(PER_SUBJECT[(n_way, cond)]) * 100
    b = np.array(PER_SUBJECT[(n_way, control)]) * 100
    d = a - b
    n = len(d)
    sd = d.std(ddof=1)
    se = sd / np.sqrt(n)
    t_stat, p = stats.ttest_rel(a, b)
    tcrit = stats.t.ppf(0.975, n - 1)
    return dict(
        n_way=n_way, contrast=f"{cond} - {control}",
        delta_pp=d.mean(), sd_pp=sd, se_pp=se,
        ci_lo=d.mean() - tcrit * se, ci_hi=d.mean() + tcrit * se,
        dz=d.mean() / sd, t=t_stat, p=p,
        favouring=f"{int((d > 0).sum())}/{n}",
    )

contrasts = pd.DataFrame([
    paired_contrast(n, c) for n in N_WAYS for c in ("emg_only", "imu_only")
])

show = contrasts.copy()
show["95% CI"] = show.apply(lambda r: f"[{r.ci_lo:+6.1f}, {r.ci_hi:+6.1f}]", axis=1)
print(show[["n_way", "contrast", "delta_pp", "95% CI", "dz", "t", "p", "favouring"]]
      .to_string(index=False, formatters={"delta_pp": "{:+.2f}".format,
                                          "dz": "{:+.2f}".format,
                                          "t": "{:+.2f}".format,
                                          "p": "{:.3f}".format}))
print("\n'favouring' = number of subjects (out of 4) where the condition beat the control.")
print("Note: d_z is upward-biased at n = 4 and should not be quoted as a precise effect size.")

 n_way        contrast delta_pp           95% CI    dz     t     p favouring
     3 emg_only - both    +1.16 [  -4.3,   +6.7] +0.33 +0.67 0.552       3/4
     3 imu_only - both    +2.84 [  -2.6,   +8.2] +0.84 +1.67 0.193       4/4
     5 emg_only - both    -8.51 [ -16.1,   -0.9] -1.77 -3.55 0.038       0/4
     5 imu_only - both    +1.19 [  -5.9,   +8.3] +0.27 +0.53 0.631       3/4
    10 emg_only - both   -18.14 [ -28.5,   -7.7] -2.77 -5.54 0.012       0/4
    10 imu_only - both    +2.05 [ -10.8,  +14.9] +0.25 +0.51 0.646       3/4

'favouring' = number of subjects (out of 4) where the condition beat the control.
Note: d_z is upward-biased at n = 4 and should not be quoted as a precise effect size.


## 6. Multiplicity correction

Two defensible family definitions, reported side by side because the choice changes the
conclusion and should be stated openly rather than picked quietly.

- **Family = 2 per n_way.** Treats each vocabulary size as its own inferential question,
  with 2 contrasts against a common control. Justified by the significant interaction.
- **Family = all 6.** Treats the whole table as one exploratory sweep. Strictly more
  conservative.

In [9]:
def holm(pvals):
    """Holm-Bonferroni step-down adjusted p-values, with monotonicity enforced."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for i, idx in enumerate(order):
        running = max(running, min(1.0, p[idx] * (m - i)))
        adj[idx] = running
    return adj

contrasts["holm_within_nway"] = np.nan
for n in N_WAYS:
    m = contrasts.n_way == n
    contrasts.loc[m, "holm_within_nway"] = holm(contrasts.loc[m, "p"].values)

contrasts["holm_all_six"] = holm(contrasts["p"].values)

out = contrasts[["n_way", "contrast", "delta_pp", "p", "holm_within_nway", "holm_all_six"]].copy()
out["verdict (family=2)"] = np.where(out.holm_within_nway < 0.05, "significant", "n.s.")
print(out.to_string(index=False, formatters={"delta_pp": "{:+.2f}".format,
                                             "p": "{:.3f}".format,
                                             "holm_within_nway": "{:.3f}".format,
                                             "holm_all_six": "{:.3f}".format}))

print("\nSurvives Holm at family = 2 per n_way :",
      ", ".join(f"{r.n_way}-way {r.contrast}" for r in contrasts.itertuples()
                if r.holm_within_nway < 0.05) or "none")
print("Survives Holm at family = all 6       :",
      ", ".join(f"{r.n_way}-way {r.contrast}" for r in contrasts.itertuples()
                if r.holm_all_six < 0.05) or "none")

 n_way        contrast delta_pp     p holm_within_nway holm_all_six verdict (family=2)
     3 emg_only - both    +1.16 0.552            0.552        1.000               n.s.
     3 imu_only - both    +2.84 0.193            0.385        0.770               n.s.
     5 emg_only - both    -8.51 0.038            0.076        0.190               n.s.
     5 imu_only - both    +1.19 0.631            0.631        1.000               n.s.
    10 emg_only - both   -18.14 0.012            0.023        0.069        significant
    10 imu_only - both    +2.05 0.646            0.646        1.000               n.s.

Survives Holm at family = 2 per n_way : 10-way emg_only - both
Survives Holm at family = all 6       : none


## 7. Why not Wilcoxon signed-rank

The usual objection to a t-test at n = 4 is that normality is unverifiable, and the usual
remedy is a nonparametric test. That remedy does not work here, and it is worth being able
to say why in one sentence.

With 4 paired observations there are only 2^4 = 16 possible sign assignments, so the smallest
attainable one-sided p is 1/16 and the smallest two-sided p is 1/8 = 0.125. **Wilcoxon
signed-rank cannot return p < 0.05 at this sample size regardless of effect magnitude.**
The same holds for the sign test.

In [10]:
n = 4
print(f"n = {n}: sign assignments = 2^{n} = {2**n}")
print(f"  minimum attainable one-sided p = {1/2**n:.4f}")
print(f"  minimum attainable two-sided p = {2/2**n:.4f}   <- above the 0.05 threshold\n")

print("Wilcoxon applied to the largest observed effect (10-way, EMG vs fused):")
a = np.array(PER_SUBJECT[(10, "emg_only")]) * 100
b = np.array(PER_SUBJECT[(10, "both")]) * 100
w_stat, w_p = stats.wilcoxon(a, b, alternative="two-sided", zero_method="wilcox",
                             mode="exact" if "mode" in stats.wilcoxon.__code__.co_varnames
                             else "auto")
print(f"  differences (pp): {np.round(a - b, 2)}")
print(f"  all four in the same direction, mean {np.mean(a-b):+.2f} pp")
print(f"  Wilcoxon W = {w_stat:.1f}, p = {w_p:.4f}  <- floored, cannot go lower")
print(f"  paired t-test for comparison: p = {stats.ttest_rel(a, b).pvalue:.4f}")
print("\n-> Wilcoxon is not wrong here, it simply has no resolution. Reject it on")
print("   power grounds, not validity grounds.")

n = 4: sign assignments = 2^4 = 16
  minimum attainable one-sided p = 0.0625
  minimum attainable two-sided p = 0.1250   <- above the 0.05 threshold

Wilcoxon applied to the largest observed effect (10-way, EMG vs fused):
  differences (pp): [-15.56 -16.7  -12.68 -27.62]
  all four in the same direction, mean -18.14 pp
  Wilcoxon W = 0.0, p = 0.1250  <- floored, cannot go lower
  paired t-test for comparison: p = 0.0116

-> Wilcoxon is not wrong here, it simply has no resolution. Reject it on
   power grounds, not validity grounds.


## 8. Minimum detectable effect

Converts "not significant" into a concrete statement about what this design could and could
not have seen. This is the number to quote instead of interpreting a null result as evidence
of no effect.

In [11]:
from statsmodels.stats.power import TTestPower

dz_needed = TTestPower().solve_power(effect_size=None, nobs=4, alpha=0.05,
                                     power=0.80, alternative="two-sided")
print(f"Paired design, n = 4, alpha = 0.05, power = 0.80")
print(f"  required effect size d_z = {dz_needed:.2f}\n")
print("Translated into percentage points, using each contrast's observed SD:\n")
for r in contrasts.itertuples():
    mde = dz_needed * r.sd_pp
    detectable = "DETECTABLE" if abs(r.delta_pp) >= mde else "below MDE"
    print(f"  {r.n_way:>2}-way {r.contrast:22s} SD={r.sd_pp:5.2f}  "
          f"MDE={mde:6.2f} pp   observed={r.delta_pp:+7.2f} pp   {detectable}")

imu = contrasts[contrasts.contrast.str.startswith("imu")]
print(f"\n-> Across the three IMU-vs-fused contrasts the MDE ranges "
      f"{(dz_needed*imu.sd_pp).min():.1f} to {(dz_needed*imu.sd_pp).max():.1f} pp.")
print("   A genuine fusion benefit of a few points would be invisible to this design.")
print("   'Not significant' here means UNDERPOWERED, not 'no effect'.")
print("")
print("   Note: a contrast can reach p < 0.05 while sitting below the 80%-power MDE")
print("   (5-way EMG does). MDE describes the design's typical sensitivity, not a")
print("   threshold individual results must clear.")

Paired design, n = 4, alpha = 0.05, power = 0.80
  required effect size d_z = 2.13

Translated into percentage points, using each contrast's observed SD:

   3-way emg_only - both        SD= 3.46  MDE=  7.36 pp   observed=  +1.16 pp   below MDE
   3-way imu_only - both        SD= 3.40  MDE=  7.23 pp   observed=  +2.84 pp   below MDE
   5-way emg_only - both        SD= 4.79  MDE= 10.20 pp   observed=  -8.51 pp   below MDE
   5-way imu_only - both        SD= 4.46  MDE=  9.49 pp   observed=  +1.19 pp   below MDE
  10-way emg_only - both        SD= 6.54  MDE= 13.92 pp   observed= -18.14 pp   DETECTABLE
  10-way imu_only - both        SD= 8.08  MDE= 17.19 pp   observed=  +2.05 pp   below MDE

-> Across the three IMU-vs-fused contrasts the MDE ranges 7.2 to 17.2 pp.
   A genuine fusion benefit of a few points would be invisible to this design.
   'Not significant' here means UNDERPOWERED, not 'no effect'.

   Note: a contrast can reach p < 0.05 while sitting below the 80%-power MDE
   (5-way

## 9. Validation vs. test

The two four-subject draws are disjoint (`val_PIDs` = P005/P131/P116/P109,
`test_PIDs` = P004/P104/P105/P121) and disagree in **sign** on the fusion question.

`best_val_acc` is a maximum over epochs on the set used for checkpoint selection, so it is
already optimistically biased — and test still exceeds it in every cell. That makes the
subject-difficulty gap larger than the raw numbers suggest.

In [12]:
rows = []
for n in N_WAYS:
    for c in CONDITIONS:
        te, va = REPORTED[(n, c)]
        rows.append(dict(n_way=n, cond=c, test=te*100, val=va*100, gap=(te-va)*100))
vt = pd.DataFrame(rows)

print("Test accuracy (%)\n")
print(vt.pivot(index="n_way", columns="cond", values="test").round(2).to_string())
print("\nValidation accuracy, best epoch (%)\n")
print(vt.pivot(index="n_way", columns="cond", values="val").round(2).to_string())
print("\nTest - validation (pp)\n")
print(vt.pivot(index="n_way", columns="cond", values="gap").round(2).to_string())

print("\nWinner by split:")
for n in N_WAYS:
    sub = vt[vt.n_way == n]
    print(f"  {n:>2}-way   test -> {sub.loc[sub.test.idxmax(),'cond']:9s}"
          f"   val -> {sub.loc[sub.val.idxmax(),'cond']}")
print("\n-> Validation favours the fused model at every n_way; test favours imu_only at")
print("   every n_way. Two 4-subject draws, opposite answers. This is the clearest")
print("   available evidence that n = 4 cannot settle the fusion question.")
print(f"\nTest exceeds validation in {int((vt.gap > 0).sum())}/9 cells "
      f"(mean gap {vt.gap.mean():+.2f} pp).")

Test accuracy (%)

cond     both  emg_only  imu_only
n_way                            
3     87.5800   88.7400   90.4200
5     80.2900   71.7900   81.4800
10    63.3100   45.1700   65.3600

Validation accuracy, best epoch (%)

cond     both  emg_only  imu_only
n_way                            
3     87.3800   78.7500   85.7700
5     78.9200   66.0300   75.4400
10    61.6500   41.8900   59.6800

Test - validation (pp)

cond    both  emg_only  imu_only
n_way                           
3     0.2000    9.9900    4.6600
5     1.3800    5.7600    6.0400
10    1.6600    3.2800    5.6800

Winner by split:
   3-way   test -> imu_only    val -> both
   5-way   test -> imu_only    val -> both
  10-way   test -> imu_only    val -> both

-> Validation favours the fused model at every n_way; test favours imu_only at
   every n_way. Two 4-subject draws, opposite answers. This is the clearest
   available evidence that n = 4 cannot settle the fusion question.

Test exceeds validation in 9/9 cells (mea

## 10. Summary

Printed conclusions, generated from the numbers above rather than typed by hand.

In [13]:
print("=" * 78)
print("A13 MODALITY ABLATION - CONCLUSIONS")
print("=" * 78)

tbl = aov.anova_table
print(f"\n1. OMNIBUS")
print(f"   condition x n_way interaction: F(4,12) = {tbl.loc['cond:n_way','F Value']:.2f}, "
      f"p = {tbl.loc['cond:n_way','Pr > F']:.4f}")
print(f"   -> the modality effect depends on vocabulary size; decompose by n_way.")

print(f"\n2. EMG-ONLY")
for r in contrasts[contrasts.contrast.str.startswith("emg")].itertuples():
    sig = "significant" if r.holm_within_nway < 0.05 else "n.s."
    print(f"   {r.n_way:>2}-way: {r.delta_pp:+7.2f} pp vs fused, Holm p = {r.holm_within_nway:.3f} ({sig}), "
          f"{r.favouring} subjects")
print("   -> EMG alone cannot carry the task as the label set grows.")

print(f"\n3. IMU-ONLY")
for r in contrasts[contrasts.contrast.str.startswith("imu")].itertuples():
    sig = "significant" if r.holm_within_nway < 0.05 else "n.s."
    print(f"   {r.n_way:>2}-way: {r.delta_pp:+7.2f} pp vs fused, Holm p = {r.holm_within_nway:.3f} ({sig}), "
          f"CI [{r.ci_lo:+.1f}, {r.ci_hi:+.1f}]")
print("   -> indistinguishable from fused everywhere; every CI contains zero.")
print("   -> we CANNOT claim fusion helps. We also cannot claim it hurts.")

print(f"\n4. POWER")
print(f"   MDE at n=4, 80% power: {dz_needed:.2f} d_z "
      f"= {(dz_needed*imu.sd_pp).min():.1f}-{(dz_needed*imu.sd_pp).max():.1f} pp for the IMU contrasts.")

print(f"\n5. SPLIT SENSITIVITY")
print(f"   Validation favours fused at all 3 n_way; test favours imu_only at all 3.")
print(f"   Test exceeds validation in {int((vt.gap>0).sum())}/9 cells.")

print("\n" + "=" * 78)
print("BOTTOM LINE: IMU is the dominant modality. EMG alone degrades sharply with")
print("vocabulary size. Fusion is not measurably better than IMU alone on this split,")
print("and four subjects cannot resolve a difference of this magnitude either way.")
print("=" * 78)

A13 MODALITY ABLATION - CONCLUSIONS

1. OMNIBUS
   condition x n_way interaction: F(4,12) = 19.23, p = 0.0000
   -> the modality effect depends on vocabulary size; decompose by n_way.

2. EMG-ONLY
    3-way:   +1.16 pp vs fused, Holm p = 0.552 (n.s.), 3/4 subjects
    5-way:   -8.51 pp vs fused, Holm p = 0.076 (n.s.), 0/4 subjects
   10-way:  -18.14 pp vs fused, Holm p = 0.023 (significant), 0/4 subjects
   -> EMG alone cannot carry the task as the label set grows.

3. IMU-ONLY
    3-way:   +2.84 pp vs fused, Holm p = 0.385 (n.s.), CI [-2.6, +8.2]
    5-way:   +1.19 pp vs fused, Holm p = 0.631 (n.s.), CI [-5.9, +8.3]
   10-way:   +2.05 pp vs fused, Holm p = 0.646 (n.s.), CI [-10.8, +14.9]
   -> indistinguishable from fused everywhere; every CI contains zero.
   -> we CANNOT claim fusion helps. We also cannot claim it hurts.

4. POWER
   MDE at n=4, 80% power: 2.13 d_z = 7.2-17.2 pp for the IMU contrasts.

5. SPLIT SENSITIVITY
   Validation favours fused at all 3 n_way; test favours imu

---

### Optional: linear mixed model

Uses all 36 cells at once rather than nine separate 4-point comparisons, giving tighter
intervals. **Hold this in reserve rather than leading with it:** the 36 values come from nine
single-seed runs, so the model treats run-to-run variance as residual and will understate
uncertainty. Useful if someone asks whether a more powerful analysis is available.

In [14]:
import statsmodels.formula.api as smf

df_lmm = df.copy()
df_lmm["cond"] = pd.Categorical(df_lmm["cond"], categories=["both", "emg_only", "imu_only"])
df_lmm["n_way_c"] = pd.Categorical(df_lmm["n_way"], categories=[3, 5, 10])

lmm = smf.mixedlm("acc ~ C(cond) * C(n_way_c)", df_lmm, groups=df_lmm["subject"]).fit()
print(lmm.summary())
print("\nCoefficients are relative to (cond=both, n_way=3).")
print("CAVEAT: single seed per cell - residual variance conflates subject noise with")
print("run-to-run variance. Treat these intervals as optimistic.")

                      Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          acc     
No. Observations:           36               Method:                      REML    
No. Groups:                 4                Scale:                       19.8234 
Min. group size:            9                Log-Likelihood:              -88.9660
Max. group size:            9                Converged:                   Yes     
Mean group size:            9.0                                                   
----------------------------------------------------------------------------------
                                      Coef.  Std.Err.   z    P>|z|  [0.025  0.975]
----------------------------------------------------------------------------------
Intercept                             87.580    3.583 24.440 0.000  80.556  94.603
C(cond)[T.emg_only]                    1.156    3.148  0.367 0.714  -5.015   7.326
C(cond)[T.imu_only]        